# 2 · Physics agent with the GAME framework

The pendulum agent of [`1-physics_agent_pendulum.ipynb`](1-physics_agent_pendulum.ipynb), rebuilt with the GAME framework of [`course/3-building_agent_with_GAME.ipynb`](../course/3-building_agent_with_GAME.ipynb).

**Why:** in notebook 1, MiniCPM reasoned correctly but wrote `<action>` tags instead of a ```` ```action ```` block; the strict parser rejected every answer and the agent never called a tool.

| | Notebook 1 | This notebook |
|---|---|---|
| Format errors | strict parser → same error forever | tolerant parser (```` ```action ````, `<action>`, ```` ```json ````) + **one retry** with an actionable message |
| Tools | `if/elif` chain in the loop | `Action`s in an `ActionRegistry` |
| Wrong argument / unknown tool | error message | error returned to the agent (`Environment`); unknown tool names trigger the retry, with `SafeAgent` as a safety net |
| Instructions | one long system prompt | `Goal` objects |
| Loop | written by hand | `Agent.run()` from the framework, unchanged |

**Task:** *find the amplitude at which the period is 10% longer than the small-angle period*, the task that failed in notebook 1.

## Setup

In [ ]:
import os, json, time, traceback
from dataclasses import dataclass, field
from typing import List, Callable, Dict, Any

import numpy as np
from scipy.integrate import solve_ivp      # ODE integrator (physics tools)
from scipy.special import ellipk           # exact period (only to check the results)
from scipy.optimize import brentq          # root finding (only to check the results)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Load MiniCPM (~5 GB, downloaded on first run, then cached)
model_id = "openbmb/MiniCPM5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

## GAME framework

Copied **unchanged** from `course/3-building_agent_with_GAME.ipynb` (see that notebook and `3-GAME_framework_notes.md` for explanations):
- `generate_response`: MiniCPM with emulated function calling (tolerant parser + retry),
- `Goal`, `Action`, `ActionRegistry`, `Memory`, `Environment`,
- `AgentFunctionCallingActionLanguage`, `Agent`.

In [ ]:
@dataclass
class Prompt:
    messages: List[Dict] = field(default_factory=list)   # goals + memory, as chat messages
    tools: List[Dict] = field(default_factory=list)      # actions, as JSON schemas
    metadata: dict = field(default_factory=dict)         # default_factory avoids a shared mutable default


def minicpm_chat(messages: List[Dict], max_new_tokens: int = 1024) -> str:
    """Send chat messages to MiniCPM and return its reply as text (same helper as notebooks 0-2)."""
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # skip MiniCPM's hidden reasoning step
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():             # no gradients: we are not training
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Decode only the new tokens (drop the prompt)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def tools_to_instructions(tools: List[Dict]) -> str:
    """Describe the tools in plain text, since MiniCPM has no function-calling API."""
    tool_list = [
        {
            "name": t["function"]["name"],
            "description": t["function"]["description"],
            "parameters": t["function"]["parameters"],
        }
        for t in tools
    ]
    return (
        "You can use the following tools:\n"
        "```json\n" + json.dumps(tool_list, indent=2) + "\n```\n\n"
        "Every response MUST call exactly one tool. First think step by step, "
        "then write the tool call as a Markdown code block in exactly this format "
        "(a code block, not XML tags):\n"
        "```action\n"
        '{"tool": "<tool name>", "args": {<arguments>}}\n'
        "```"
    )


def extract_tool_call(text: str, tool_names: List[str]):
    """Find a tool call in the model's text. Returns {"tool": ..., "args": ...} or None.

    Tolerant: tries ```action, <action>...</action>, ```json and finally the whole text.
    """
    candidates = []
    if "```action" in text:
        candidates.append(text.split("```action", 1)[1].split("```", 1)[0])
    if "<action>" in text:
        candidates.append(text.split("<action>", 1)[1].split("</action>", 1)[0])
    if "```json" in text:
        candidates.append(text.split("```json", 1)[1].split("```", 1)[0])
    candidates.append(text)

    for candidate in candidates:
        try:
            data = json.loads(candidate.strip())
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict):
            name = data.get("tool") or data.get("tool_name") or data.get("name")
            if name in tool_names:
                return {"tool": name, "args": data.get("args") or data.get("arguments") or {}}
    return None


def generate_response(prompt: Prompt, max_retries: int = 1) -> str:
    """Call the LLM. Returns a tool call as a JSON string, or plain text.

    Same output as the course's GPT-4o version:
        '{"tool": "<name>", "args": {...}}'  if the model called a tool,
        the model's text                      otherwise.
    """
    messages = [dict(m) for m in prompt.messages]          # copy: don't modify the prompt

    # No tools: just chat
    if not prompt.tools:
        return minicpm_chat(messages)

    # 1. Tools -> text instructions, appended to the system message
    instructions = tools_to_instructions(prompt.tools)
    if messages and messages[0]["role"] == "system":
        messages[0]["content"] += "\n\n" + instructions
    else:
        messages.insert(0, {"role": "system", "content": instructions})

    tool_names = [t["function"]["name"] for t in prompt.tools]

    for attempt in range(max_retries + 1):
        # 2. Generate
        text = minicpm_chat(messages)

        # 3. Extract the tool call
        call = extract_tool_call(text, tool_names)
        if call is not None:
            return json.dumps(call)

        # 4. Retry with an actionable error message
        messages += [
            {"role": "assistant", "content": text},
            {"role": "user", "content":
                "Your answer did not contain a valid tool call. Available tools: "
                + ", ".join(tool_names) + ". Reply with one tool call in this format:\n"
                '```action\n{"tool": "<tool name>", "args": {...}}\n```'},
        ]

    return text   # still no tool call: return the text (the agent language will handle it)

In [ ]:
@dataclass(frozen=True)          # frozen: a goal is immutable once created
class Goal:
    priority: int                # order of importance (used to sort/format goals)
    name: str                    # short label
    description: str             # what to achieve and how


class Action:
    """One tool the agent can use: its Python function + the description the LLM sees."""

    def __init__(self,
                 name: str,               # name the LLM uses to call it
                 function: Callable,      # Python function that does the work
                 description: str,        # what it does (shown to the LLM)
                 parameters: Dict,        # JSON schema of the arguments (shown to the LLM)
                 terminal: bool = False): # True -> running it ends the agent loop
        self.name = name
        self.function = function
        self.description = description
        self.terminal = terminal
        self.parameters = parameters

    def execute(self, **args) -> Any:
        """Execute the action's function"""
        return self.function(**args)


class ActionRegistry:
    """Lookup table name -> Action."""

    def __init__(self):
        self.actions = {}

    def register(self, action: Action):
        self.actions[action.name] = action

    def get_action(self, name: str) -> [Action, None]:
        return self.actions.get(name, None)          # None if the LLM invents a tool name

    def get_actions(self) -> List[Action]:
        """Get all registered actions"""
        return list(self.actions.values())


class Memory:
    """The agent's history: a list of dicts like {"type": "assistant", "content": "..."}."""

    def __init__(self):
        self.items = []  # basic conversation history

    def add_memory(self, memory: dict):
        """Add memory to working memory"""
        self.items.append(memory)

    def get_memories(self, limit: int = None) -> List[Dict]:
        """Get formatted conversation history for prompt"""
        # Note: items[:limit] returns the FIRST `limit` items. To get the most
        # recent ones (as the course text describes), it would be items[-limit:].
        # With limit=None (the default used here) both return everything.
        return self.items[:limit]

    def copy_without_system_memories(self):
        """Return a copy of the memory without system memories"""
        filtered_items = [m for m in self.items if m["type"] != "system"]
        memory = Memory()
        memory.items = filtered_items
        return memory


class Environment:
    """Runs actions and wraps their results (or errors) in a standard dict."""

    def execute_action(self, action: Action, args: dict) -> dict:
        """Execute an action and return the result."""
        try:
            result = action.execute(**args)
            return self.format_result(result)
        except Exception as e:
            # The error goes back to the agent instead of crashing the loop
            return {
                "tool_executed": False,
                "error": str(e),
                "traceback": traceback.format_exc()
            }

    def format_result(self, result: Any) -> dict:
        """Format the result with metadata."""
        return {
            "tool_executed": True,
            "result": result,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")
        }

In [ ]:
class AgentLanguage:
    """Interface: how GAME objects become a prompt, and how the reply becomes an action."""

    def __init__(self):
        pass

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        raise NotImplementedError("Subclasses must implement this method")

    def parse_response(self, response: str) -> dict:
        raise NotImplementedError("Subclasses must implement this method")


class AgentFunctionCallingActionLanguage(AgentLanguage):
    """Language for function calling: goals -> system message, actions -> tool schemas."""

    def __init__(self):
        super().__init__()

    def format_goals(self, goals: List[Goal]) -> List:
        # All goals -> one system message: "name:\n----\ndescription\n----"
        sep = "\n-------------------\n"
        goal_instructions = "\n\n".join([f"{goal.name}:{sep}{goal.description}{sep}" for goal in goals])
        return [
            {"role": "system", "content": goal_instructions}
        ]

    def format_memory(self, memory: Memory) -> List:
        """Map memory items to chat messages."""
        # assistant decisions -> role "assistant"
        # environment results -> role "user"   (changed for MiniCPM, see above)
        # user messages       -> role "user"
        items = memory.get_memories()
        mapped_items = []
        for item in items:

            content = item.get("content", None)
            if not content:
                content = json.dumps(item, indent=4)

            if item["type"] == "assistant":
                mapped_items.append({"role": "assistant", "content": content})
            elif item["type"] == "environment":
                mapped_items.append({"role": "user", "content": content})
            else:
                mapped_items.append({"role": "user", "content": content})

        return mapped_items

    def format_actions(self, actions: List[Action]) -> [List, List]:
        """Actions -> tool schemas in the standard function-calling format."""
        tools = [
            {
                "type": "function",
                "function": {
                    "name": action.name,
                    # Include up to 1024 characters of the description
                    "description": action.description[:1024],
                    "parameters": action.parameters,
                },
            } for action in actions
        ]

        return tools

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        # messages = goals (system) + memory (conversation); tools = actions
        prompt = []
        prompt += self.format_goals(goals)
        prompt += self.format_memory(memory)

        tools = self.format_actions(actions)

        return Prompt(messages=prompt, tools=tools)

    def adapt_prompt_after_parsing_error(self,
                                         prompt: Prompt,
                                         response: str,
                                         traceback: str,
                                         error: Any,
                                         retries_left: int) -> Prompt:
        # Hook for later: repair the prompt after a parsing error (unused for now)
        return prompt

    def parse_response(self, response: str) -> dict:
        """Parse the LLM response: a JSON tool call, or plain text -> terminate."""
        try:
            return json.loads(response)

        except Exception as e:
            # Not a tool call: treat the text as the final answer
            return {
                "tool": "terminate",
                "args": {"message": response}
            }

In [ ]:
class Agent:
    def __init__(self,
                 goals: List[Goal],
                 agent_language: AgentLanguage,
                 action_registry: ActionRegistry,
                 generate_response: Callable[[Prompt], str],
                 environment: Environment):
        """
        Initialize an agent with its core GAME components
        """
        self.goals = goals                          # G
        self.generate_response = generate_response  # the LLM
        self.agent_language = agent_language        # GAME <-> prompt translator
        self.actions = action_registry              # A
        self.environment = environment              # E   (M is passed to run())

    def construct_prompt(self, goals: List[Goal], memory: Memory, actions: ActionRegistry) -> Prompt:
        """Build prompt with memory context"""
        return self.agent_language.construct_prompt(
            actions=actions.get_actions(),
            environment=self.environment,
            goals=goals,
            memory=memory
        )

    def get_action(self, response):
        # Reply -> {"tool": ..., "args": ...} -> the registered Action object
        invocation = self.agent_language.parse_response(response)
        action = self.actions.get_action(invocation["tool"])
        return action, invocation

    def should_terminate(self, response: str) -> bool:
        action_def, _ = self.get_action(response)
        return action_def.terminal

    def set_current_task(self, memory: Memory, task: str):
        memory.add_memory({"type": "user", "content": task})

    def update_memory(self, memory: Memory, response: str, result: dict):
        """
        Update memory with the agent's decision and the environment's response.
        """
        new_memories = [
            {"type": "assistant", "content": response},
            {"type": "environment", "content": json.dumps(result)}
        ]
        for m in new_memories:
            memory.add_memory(m)

    def prompt_llm_for_action(self, full_prompt: Prompt) -> str:
        response = self.generate_response(full_prompt)
        return response

    def run(self, user_input: str, memory=None, max_iterations: int = 50) -> Memory:
        """
        Execute the GAME loop for this agent with a maximum iteration limit.
        """
        memory = memory or Memory()
        self.set_current_task(memory, user_input)

        for _ in range(max_iterations):
            # 1. Construct a prompt that includes the Goals, Actions, and the current Memory
            prompt = self.construct_prompt(self.goals, memory, self.actions)

            # 2. Generate a response from the agent
            print("Agent thinking...")
            response = self.prompt_llm_for_action(prompt)
            print(f"Agent Decision: {response}")

            # 3. Determine which action the agent wants to execute
            action, invocation = self.get_action(response)

            # 4. Execute the action in the environment
            result = self.environment.execute_action(action, invocation["args"])
            print(f"Action Result: {result}")

            # 5. Update the agent's memory with information about what happened
            self.update_memory(memory, response, result)

            # 6. Check if the agent has decided to terminate
            if self.should_terminate(response):
                break

        return memory

### `SafeAgent`: one addition

In the course `Agent`, an invented tool name makes `get_action` return `None` and the loop crashes in `should_terminate`. `generate_response` already retries when the tool name is unknown, but if the retry fails too, `SafeAgent` replaces the missing action with one that returns an error message → the loop never crashes. Extending the framework by **subclassing**, without editing it.

In [ ]:
class SafeAgent(Agent):
    """Agent that never crashes on an unknown tool name or a malformed invocation."""

    def get_action(self, response):
        invocation = self.agent_language.parse_response(response)

        # Malformed invocation (e.g. valid JSON without "tool") -> treat the text as the final answer
        if not isinstance(invocation, dict) or "tool" not in invocation:
            invocation = {"tool": "terminate", "args": {"message": str(response)}}
        invocation.setdefault("args", {})

        action = self.actions.get_action(invocation["tool"])
        if action is None:
            # Unknown tool -> an action whose result tells the agent what went wrong
            name = invocation["tool"]
            available = [a.name for a in self.actions.get_actions()]
            action = Action(
                name="unknown_tool",
                function=lambda **_: f"Error: unknown tool '{name}'. Available tools: {available}",
                description="", parameters={}, terminal=False,
            )
        return action, invocation

## Physics tools

Same tools as notebook 1: nonlinear pendulum $\ddot\theta = -\frac{g}{L}\sin\theta$ integrated with `solve_ivp`; the period is measured from the times $\theta = 0$; $T_0 = 2\pi\sqrt{L/g}$ for small oscillations. Results are short strings; trajectories stay in `simulations`.

In [ ]:
G = 9.81            # gravitational acceleration [m/s^2]
simulations = {}    # simulation results live here (NOT in the LLM's memory)


def simulate_pendulum(theta0_deg: float, length_m: float = 1.0, t_max: float = 20.0) -> str:
    """Integrate the NONLINEAR pendulum equation  θ'' = -(g/L) sin θ.

    The pendulum starts at rest from angle theta0_deg. The trajectory is stored
    in `simulations` and only an id (e.g. "sim_1") is returned to the LLM.
    """
    # Arguments come from the LLM: convert and check them
    theta0_deg, length_m, t_max = float(theta0_deg), float(length_m), float(t_max)
    if not 0 < theta0_deg < 180:
        return "Error: theta0_deg must be between 0 and 180 degrees (excluded)."
    if length_m <= 0 or t_max <= 0:
        return "Error: length_m and t_max must be positive."

    # First-order system: y = [θ, ω]  ->  y' = [ω, -(g/L) sin θ]
    def rhs(t, y):
        return [y[1], -(G / length_m) * np.sin(y[0])]

    # Event: θ = 0 (crossing the vertical), used to measure the period
    def crosses_vertical(t, y):
        return y[0]

    sol = solve_ivp(
        rhs, [0, t_max], [np.radians(theta0_deg), 0.0],   # initial conditions: θ0, ω = 0
        events=crosses_vertical,
        rtol=1e-9, atol=1e-9,                             # high accuracy
    )

    sim_id = f"sim_{len(simulations) + 1}"
    simulations[sim_id] = {"theta0_deg": theta0_deg, "length_m": length_m, "solution": sol}
    return f"{sim_id}: theta0 = {theta0_deg:g}°, L = {length_m:g} m, simulated for {t_max:g} s"


def measure_period(sim_id: str) -> str:
    """Measure the period from the times the pendulum crosses the vertical."""
    if sim_id not in simulations:
        return f"Error: unknown sim_id '{sim_id}'. Available: {list(simulations)}"

    sim = simulations[sim_id]
    crossings = sim["solution"].t_events[0]          # times when θ = 0
    if len(crossings) < 3:
        return f"Error: {sim_id} is too short to measure the period. Simulate a longer t_max."

    period = 2 * np.mean(np.diff(crossings))         # 2 consecutive crossings = half a period
    return f"{sim_id}: theta0 = {sim['theta0_deg']:g}°, measured period = {period:.4f} s"


def small_angle_period(length_m: float = 1.0) -> str:
    """Period in the harmonic (small-angle) approximation: T0 = 2π √(L/g)."""
    length_m = float(length_m)
    if length_m <= 0:
        return "Error: length_m must be positive."
    return f"small-angle period T0 = {2 * np.pi * np.sqrt(length_m / G):.4f} s (L = {length_m:g} m)"

### Test the tools against the exact solution

In [ ]:
def exact_period(theta0_deg, length_m=1.0):
    """Exact period of the nonlinear pendulum (elliptic integral) - only for checking."""
    m = np.sin(np.radians(theta0_deg) / 2) ** 2
    return 4 / np.sqrt(G / length_m) * ellipk(m)

print(small_angle_period())
for angle in [10, 45, 80, 120]:
    sim = simulate_pendulum(angle)
    sim_id = sim.split(":")[0]
    print(f"{measure_period(sim_id)} | exact = {exact_period(angle):.4f} s")

simulations.clear()   # the agent starts from sim_1

## G · Goals

The long system prompt of notebook 1, split into goals: **role**, **method**, **termination**.

In [ ]:
goals = [
    Goal(priority=1, name="Role",
         description="You are a physics assistant that answers questions about the simple pendulum "
                     "by running numerical tools. Never do calculations in your head: always use the tools."),
    Goal(priority=2, name="Method",
         description="For each amplitude: first call simulate_pendulum, then measure_period with the returned sim_id. "
                     "Use small_angle_period to get the small-angle reference T0. "
                     "To find an amplitude with a given property, try several amplitudes and narrow the range. "
                     "Use only one tool per response."),
    Goal(priority=3, name="Terminate",
         description="When you have the answer, call terminate with a message that reports the numbers "
                     "you measured and explains the physics."),
]

## A · Actions

Each physics function becomes an `Action` with a JSON schema. `terminate` is the only **terminal** action.

In [ ]:
action_registry = ActionRegistry()

action_registry.register(Action(
    name="simulate_pendulum",
    function=simulate_pendulum,
    description="Numerically integrates the NONLINEAR pendulum equation theta'' = -(g/L) sin(theta), "
                "starting at rest from angle theta0_deg. Returns a simulation id (e.g. 'sim_1').",
    parameters={
        "type": "object",
        "properties": {
            "theta0_deg": {"type": "number", "description": "Initial angle in degrees, between 0 and 180 (excluded)."},
            "length_m": {"type": "number", "description": "Pendulum length in meters. Default 1.0."},
            "t_max": {"type": "number", "description": "Simulated time in seconds. Default 20.0."},
        },
        "required": ["theta0_deg"],
    },
))

action_registry.register(Action(
    name="measure_period",
    function=measure_period,
    description="Measures the oscillation period of a previous simulation.",
    parameters={
        "type": "object",
        "properties": {
            "sim_id": {"type": "string", "description": "Simulation id returned by simulate_pendulum, e.g. 'sim_1'."},
        },
        "required": ["sim_id"],
    },
))

action_registry.register(Action(
    name="small_angle_period",
    function=small_angle_period,
    description="Period predicted by the small-angle (harmonic) approximation T0 = 2*pi*sqrt(L/g).",
    parameters={
        "type": "object",
        "properties": {
            "length_m": {"type": "number", "description": "Pendulum length in meters. Default 1.0."},
        },
        "required": [],
    },
))

action_registry.register(Action(
    name="terminate",
    function=lambda message: message,
    description="Ends the task and gives the final answer to the user.",
    parameters={
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "Final answer: the numbers and the physical explanation."},
        },
        "required": ["message"],
    },
    terminal=True,
))

print([a.name for a in action_registry.get_actions()])

## M · E · Agent

Memory and environment are the standard ones; `run()` creates a fresh `Memory`.

In [ ]:
agent = SafeAgent(
    goals=goals,
    agent_language=AgentFunctionCallingActionLanguage(),
    action_registry=action_registry,
    generate_response=generate_response,     # MiniCPM
    environment=Environment(),
)

## Run

A search needs several simulate → measure pairs, so `max_iterations` is higher than in notebook 1.

In [ ]:
simulations.clear()

task = "Find the amplitude at which the period is 10% longer than the small-angle period (pendulum length 1 m)."
final_memory = agent.run(task, max_iterations=12)

## Check the agent's answer

Never trust the final message alone: compare the simulations the agent actually ran with the exact solution.

In [ ]:
T0 = 2 * np.pi * np.sqrt(1.0 / G)

print(f"{'sim':>6} {'theta0':>8} {'T measured':>11} {'T exact':>9} {'T/T0':>7}")
for sim_id, sim in simulations.items():
    crossings = sim["solution"].t_events[0]
    T = 2 * np.mean(np.diff(crossings)) if len(crossings) >= 3 else float("nan")
    print(f"{sim_id:>6} {sim['theta0_deg']:>7g}° {T:>10.4f}s {exact_period(sim['theta0_deg']):>8.4f}s {T / T0:>7.3f}")

# Reference answer: solve T(theta0) = 1.10 * T0 with the exact formula
answer = brentq(lambda a: exact_period(a) / T0 - 1.10, 1, 179)
print(f"\nExact answer: theta0 = {answer:.1f}°")
print(f"Iterations used: {(len(final_memory.get_memories()) - 1) // 2}")

## Results

*To fill in after running:* did the agent finish? How many iterations? How close was its answer to the exact value? Did the tolerant parser or the retry help (look for `<action>` answers in the output)?